# AA-UTE 2026

## P7.2 - Detección de anomalías con Matrix Profile (stumpy)

Este práctico muestra cómo usar **Matrix Profile** para detectar anomalías en series temporales univariadas.

Se trabajará con dos casos:

- un ejemplo con datos sintéticos;
- un ejemplo con datos del directorio `data` (formato TSB-UAD).

Luego se puede probar con los datos de demanda. 

### Trabajo a realizar

Este notebook está pensado para correr las celdas en orden y analizar resultados. Luego, se proponen ejercicios para experimentar con la ventana `m`, el tipo de anomalía y la interpretación del score.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score, average_precision_score

import stumpy

warnings.filterwarnings("ignore")
np.random.seed(42)

plt.style.use("ggplot")
plt.rcParams["figure.figsize"] = (14, 4)

%matplotlib widget

In [ ]:
def to_anomaly_score_from_mp(mp_values, n, m):
    """
    Convierte el Matrix Profile (longitud n-m+1) a un score por timestamp (longitud n).
    Un valor alto implica subsecuencia rara (discord).
    Nota: Los scores quedan desplazados a la derecha media ventana. Coinciden con el centro de la subsecuencia mientras
    que el Matrix Profile se calcula sobre la primera posición de la subsecuencia. 
    """
    mp_values = np.asarray(mp_values).reshape(-1, 1)
    score = MinMaxScaler(feature_range=(0, 1)).fit_transform(mp_values).ravel()

    left = int(np.ceil((m - 1) / 2))
    right = int((m - 1) // 2)
    return np.pad(score, (left, right), mode="edge")[:n]


def metricas_basicas(y_true, score):
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score).astype(float)
    return {
        "AUC_ROC": roc_auc_score(y_true, score),
        "AUC_PR": average_precision_score(y_true, score),
    }

## Parte 1 - Ejemplo con datos sintéticos

Construimos una señal periódica con ruido e inyectamos anomalías puntuales y por tramo.

In [ ]:
# ---------------------------------------------------------------
# 1. Serie sintética + etiquetas
# ---------------------------------------------------------------
def generar_serie_sintetica(n=4000, seed=42):
    rng = np.random.default_rng(seed)
    t = np.arange(n)

    base = 0.8 * np.sin(2 * np.pi * t / 50) + 0.3 * np.sin(2 * np.pi * t / 120)
    ruido = 0.12 * rng.standard_normal(n)
    data_syn = base + ruido
    label_syn = np.zeros(n, dtype=int)

    # Anomalías puntuales
    spikes = [600, 1450, 2500, 3300]
    data_syn[spikes] += np.array([3.2, -2.8, 3.0, -3.1])
    label_syn[spikes] = 1

    # Anomalía contextual (segmento con offset y mayor varianza)
    seg_ini, seg_fin = 2900, 3050
    data_syn[seg_ini:seg_fin] += 1.0 + 0.25 * rng.standard_normal(seg_fin - seg_ini)
    label_syn[seg_ini:seg_fin] = 1

    return data_syn, label_syn

data_syn, label_syn = generar_serie_sintetica(n=4000, seed=42)

fig, ax = plt.subplots(2, 1, figsize=(15, 5), sharex=True)
fig.suptitle("Serie sintética y etiquetas de anomalía", fontsize=14)
ax[0].plot(data_syn, color="steelblue")
ax[0].set_ylabel("valor")
ax[1].plot(label_syn, color="firebrick")
ax[1].set_ylabel("label")
ax[1].set_xlabel("tiempo")
plt.show()

In [ ]:
# ---------------------------------------------------------------
# 2. Matrix Profile con stumpy
# ---------------------------------------------------------------
m_syn = 64
mp_syn = stumpy.stump(data_syn, m=m_syn)
profile_syn = mp_syn[:, 0]
score_syn = to_anomaly_score_from_mp(profile_syn, n=len(data_syn), m=m_syn)

res_syn = metricas_basicas(label_syn, score_syn)
print("Métricas (sintético):")
for k, v in res_syn.items():
    print(f"{k}: {v:.4f}")

fig, ax = plt.subplots(4, 1, figsize=(15, 7), sharex=True)

fig.suptitle("Detección de anomalías con Matrix Profile (sintético)", fontsize=14)

ax[0].plot(data_syn, color="black")
ax[0].set_ylabel("serie")

ax[1].plot(mp_syn[:, 0], color="blue")
ax[1].set_ylabel("matrix profile")

ax[2].plot(score_syn, color="darkorange")
ax[2].set_ylabel("score")

ax[3].plot(label_syn, color="crimson")
ax[3].set_ylabel("label")
ax[3].set_xlabel("tiempo")

# share x-axis for all subplots
for a in ax:
    a.label_outer()
    


plt.show()

### Para probar (sintético)

1. Cambiar `m_syn` (por ejemplo 32, 96, 128) y comparar métricas.


## Parte 2 - Ejemplo con datos del directorio data

Se utilizará un archivo real del directorio local `data/benchmark/ECG/` incluido en el práctico.

In [ ]:
# ---------------------------------------------------------------
# 1. Carga de datos reales (TSB-UAD)
# ---------------------------------------------------------------
filepath = "data/benchmark/ECG/MBA_ECG806_data.out"
df = pd.read_csv(filepath, header=None).dropna().to_numpy()

# Para acelerar ejecución y visualización
max_len = 20000
data_real = df[:max_len, 0].astype(float)
label_real = df[:max_len, 1].astype(int)

print(f"Archivo: {filepath}")
print(f"Largo serie: {len(data_real)}")
print(f"Anomalías etiquetadas: {label_real.sum()}")

fig, ax = plt.subplots(2, 1, figsize=(15, 5), sharex=True)
fig.suptitle("Serie ECG real y etiquetas", fontsize=14)
ax[0].plot(data_real, color="steelblue")
ax[0].set_ylabel("serie")
ax[1].plot(label_real, color="firebrick")
ax[1].set_ylabel("label")
ax[1].set_xlabel("tiempo")
plt.show()

In [ ]:
# ---------------------------------------------------------------
# 2. Matrix Profile en datos reales
# ---------------------------------------------------------------
# Ventana típica para ECG en este dataset (puede ajustarse)
m_real = 100

mp_real = stumpy.stump(data_real, m=m_real)
profile_real = mp_real[:, 0]
score_real = to_anomaly_score_from_mp(profile_real, n=len(data_real), m=m_real)

res_real = metricas_basicas(label_real, score_real)
print("Métricas (real):")
for k, v in res_real.items():
    print(f"{k}: {v:.4f}")

fig, ax = plt.subplots(4, 1, figsize=(15, 7), sharex=True)
fig.suptitle("Detección de anomalías con Matrix Profile (ECG real)", fontsize=14)

ax[0].plot(data_real, color="black")
ax[0].set_ylabel("serie")

ax[1].plot(mp_real[:, 0], color="blue")
ax[1].set_ylabel("matrix profile")

ax[2].plot(score_real, color="darkorange")
ax[2].set_ylabel("score")

ax[3].plot(label_real, color="crimson")
ax[3].set_ylabel("label")
ax[3].set_xlabel("tiempo")

plt.show()

## Parte 3 - Comparación rápida

En ambos ejemplos se observa que:

- picos altos del score suelen coincidir con subsecuencias raras (discords);
- la ventana `m` controla qué patrón local se considera normal/anómalo;
- comparar métricas entre distintos `m` ayuda a calibrar el detector.

In [ ]:
pd.DataFrame([
    {"dataset": "sintetico", **res_syn, "m": m_syn},
    {"dataset": "real_ecg805", **res_real, "m": m_real},
])

### Para investigar

1. Probar otro archivo del directorio `data/benchmark/ECG` (por ejemplo `MBA_ECG820_data.out`).
2. Medir tiempo de ejecución al variar `m` y el largo de la serie.

## Parte 4 - Inspección de la función stumpy

1. ¿Qué devuelve la función __stumpy__ en cada columna?
2. ¿Qué otros parámetros tiene la función __stumpy__?

Ver: https://stumpy.readthedocs.io/en/latest/api.html#stumpy.stump


In [ ]:
mp_real.shape

In [ ]:
mp_real

### Parte 5. Datos de demanda

Probar con los datos de demanda de Mercedes  
1. Calcular el Matrix Profile
2. Ver relación con patrones repetitivos (motifs)
3. Ver relación con anomalías

In [ ]:
# ---------------------------------------------------------------
# Carga de datos (Similar al práctico 5)
# ---------------------------------------------------------------
path_file = "~/Downloads/DATOS_15MIN_MER_trainset-labeled.csv"
dta = pd.read_csv(path_file, low_memory=False)
dta["timestamp"] = pd.to_datetime(dta["timestamp"])
dta.set_index("timestamp", inplace=True)
dta = dta[dta["series"] == "OFI"].copy()
dta = dta.drop(columns=["series", "label"])
dta = dta.resample("h").mean()
dta.info()

In [ ]:
data = dta.values.astype(float)
data

In [ ]:
#TODO  Calcular el Matrix Profile y analizar,